# Concatenating different data sources
In this notebook, we will concatenate
- Cleaned VGGFace dataset
- LFW dataset
- Manually created dataset

We will use custom-pre-trained `ArcFace` to remove any duplicates in persons in all datasets using strict threshold to ensure high quality final dataset.

## 1.0 Setup the Environment

In [1]:
# !pip install deepface retinaface-pytorch

In [ ]:
# utilities
import sys
import os
import shutil
import random
import json
import time
import multiprocessing

# torch
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, ConcatDataset
import torchvision.transforms as T
import torchvision
import torch.nn.functional as F
from retinaface.pre_trained_models import get_model
from torchvision.datasets import ImageFolder
import torchvision.models as tv_models


# images/arrays
import numpy as np
import pandas as pd
import cv2
import PIL
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc
from sklearn.model_selection import train_test_split
from PIL.Image import fromarray

# custom
from src.utils.functional import load_image_cv

import tensorflow as tf
from deepface import DeepFace

2026-04-24 16:18:56.242571: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777047536.472396      80 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777047536.585575      80 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777047537.122921      80 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777047537.122974      80 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777047537.122977      80 computation_placer.cc:177] computation placer alr

26-04-24 16:19:22 - Directory /root/.deepface has been created
26-04-24 16:19:22 - Directory /root/.deepface/weights has been created


In [4]:
VGG_CLEANED_TRAIN = "/kaggle/input/datasets/ragabahmed/face-recognition-cleaned/clean/train"
VGG_CLEANED_VAL = "/kaggle/input/datasets/ragabahmed/face-recognition-cleaned/clean/val"
FAMOUS_PERSONS_DATASET = "/kaggle/input/datasets/ahmeddragabb/famous-dataset-100/100_famous_dataset"
LFW_FILTERED = "/kaggle/input/datasets/ahmeddragabb/lfw-filtered/lfw_filtered"

In [5]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

device(type='cuda')

In [6]:
gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)

for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [7]:
# detector
backbone_model = "resnet50_2020-07-20"
retina_face_detector = get_model(
    model_name = backbone_model,
    max_size = 512,
    device = DEVICE
)

retina_face_detector.eval()

Downloading: "https://github.com/ternaus/retinaface/releases/download/0.01/retinaface_resnet50_2020-07-20-f168fae3c.zip" to /root/.cache/torch/hub/checkpoints/retinaface_resnet50_2020-07-20-f168fae3c.zip


100%|██████████| 96.9M/96.9M [00:00<00:00, 189MB/s]


---
---

## 2.0 Prepare Data 

In [18]:
def get_image_paths(person_folder, max_images = 3):
    valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

    images = [
        os.path.join(person_folder, img)
        for img in os.listdir(person_folder)
        if os.path.splitext(img.lower())[1] in valid_exts
    ]

    return images[:max_images]

In [10]:
def crop_face(img_path: str):
    img = load_image_cv(img_path)

    if img is None:
        return None

    detections = retina_face_detector.predict_jsons(image = img)
    if not detections:
        return None

    bbox = detections[0].get("bbox", None)
    if bbox is None or len(bbox) != 4:
        return None

    x1, y1, x2, y2 = map(int, bbox)

    h, w = img.shape[:2]

    # clamp to image boundaries
    x1 = max(0, min(x1, w))
    x2 = max(0, min(x2, w))
    y1 = max(0, min(y1, h))
    y2 = max(0, min(y2, h))

    # invalid box
    if x2 <= x1 or y2 <= y1:
        return None

    face = img[y1:y2, x1:x2]

    if face is None or face.size == 0:
        return None

    return face

def is_valid_face(face):
    return (
        face is not None
        and hasattr(face, "size")
        and face.size > 0
        and len(face.shape) >= 2
        and face.shape[0] > 0
        and face.shape[1] > 0
    )

---
---

## 3.0 Clean datasets

In [12]:
def get_face_embedding(img_path):
    try:
        face = crop_face(img_path)

        if not is_valid_face(face):
            return None

        rep = DeepFace.represent(
            img_path = face,
            model_name = "ArcFace",
            enforce_detection = False
        )

        emb = np.array(rep[0]["embedding"], dtype=np.float32)

        # normalize embedding
        emb = emb / np.linalg.norm(emb)

        return emb

    except Exception as e:
        print(f"Embedding error: {img_path} | {e}")
        return None

In [13]:
def get_person_embedding(person_folder, max_images = 5):
    image_paths = get_image_paths(person_folder, max_images = max_images)

    embeddings = []

    for img_path in image_paths:
        emb = get_face_embedding(img_path)

        if emb is not None:
            embeddings.append(emb)

    if not embeddings:
        return None

    person_emb = np.mean(embeddings, axis = 0)
    person_emb = person_emb / np.linalg.norm(person_emb)

    return person_emb.astype(np.float32)

In [14]:
def build_original_database(original_data_path, max_images_per_person = 5):
    original_persons = [
        os.path.join(original_data_path, person)
        for person in os.listdir(original_data_path)
        if os.path.isdir(os.path.join(original_data_path, person))
    ]

    embeddings = []
    paths = []

    print(f"Building embeddings for {len(original_persons)} original persons...")
    start_time = time.time()

    for idx, person in enumerate(original_persons, start = 1):
        emb = get_person_embedding(
            person,
            max_images = max_images_per_person
        )

        if emb is not None:
            embeddings.append(emb)
            paths.append(person)

        if idx % 25 == 0:
            elapsed = round((time.time() - start_time) / 60, 3)
            print(f">> Built {idx}/{len(original_persons)} | usable: {len(paths)} | elapsed: {elapsed} min")

    if not embeddings:
        raise ValueError("No valid original embeddings found.")

    embeddings = np.vstack(embeddings).astype(np.float32)
    return embeddings, paths

In [16]:
def remove_duplicate_persons_from_multiple_sources(
    original_data_path: str,
    persons_to_add_paths: list,
    max_images_per_person: int = 5,
    similarity_threshold: float = 0.65
):
    original_embeddings, original_paths = build_original_database(
        original_data_path=original_data_path,
        max_images_per_person=max_images_per_person
    )

    cleaned_persons = list(original_paths)

    # This will grow when we add new persons
    cleaned_embeddings = original_embeddings.copy()
    cleaned_paths = list(original_paths)

    all_persons_to_add = []

    for add_path in persons_to_add_paths:
        persons = [
            os.path.join(add_path, person)
            for person in os.listdir(add_path)
            if os.path.isdir(os.path.join(add_path, person))
        ]

        all_persons_to_add.extend(persons)

    print(f"Total persons to check: {len(all_persons_to_add)}")
    print(f"Similarity threshold: {similarity_threshold}")
    print("=" * 100)

    start_time = time.time()

    for idx, person_to_add in enumerate(all_persons_to_add, start=1):
        person_name = os.path.basename(person_to_add)

        emb = get_person_embedding(
            person_to_add,
            max_images=max_images_per_person
        )

        if emb is None:
            print(f"[{idx}/{len(all_persons_to_add)}] {person_name}")
            print("Status          : skipped - no valid face")
            print("=" * 100)
            continue

        similarities = cleaned_embeddings @ emb

        best_idx = int(np.argmax(similarities))
        best_similarity = float(similarities[best_idx])
        best_match = cleaned_paths[best_idx]

        is_duplicate = best_similarity >= similarity_threshold

        if not is_duplicate:
            cleaned_persons.append(person_to_add)
            cleaned_paths.append(person_to_add)

            # add new embedding to database
            cleaned_embeddings = np.vstack([
                cleaned_embeddings,
                emb.reshape(1, -1)
            ])

        status = "DUPLICATE - not added" if is_duplicate else "NEW - added"
        elapsed = round((time.time() - start_time) / 60, 3)

        print(f"[{idx}/{len(all_persons_to_add)}] {person_name}")
        print(f"Source folder    : {os.path.basename(os.path.dirname(person_to_add))}")
        print(f"Status           : {status}")
        print(f"Best match       : {os.path.basename(best_match)}")
        print(f"Best similarity  : {best_similarity:.4f}")
        print(f"Threshold        : {similarity_threshold}")
        print(f"Total persons    : {len(cleaned_persons)}")
        print(f"Elapsed          : {elapsed} minutes")
        print("=" * 100)

    return cleaned_persons

In [19]:
original_data_path = VGG_CLEANED_TRAIN
persons_to_add_paths = [VGG_CLEANED_VAL, LFW_FILTERED, FAMOUS_PERSONS_DATASET]

cleaned_persons = remove_duplicate_persons_from_multiple_sources(
    original_data_path = original_data_path,
    persons_to_add_paths = persons_to_add_paths,
)

Building embeddings for 408 original persons...


I0000 00:00:1777048157.600493      80 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13533 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1777048157.605608      80 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


26-04-24 16:29:19 - 🔗 arcface_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/arcface_weights.h5 to /root/.deepface/weights/arcface_weights.h5...


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/arcface_weights.h5
To: /root/.deepface/weights/arcface_weights.h5
100%|██████████| 137M/137M [00:00<00:00, 205MB/s] 
I0000 00:00:1777048161.638360      80 cuda_dnn.cc:529] Loaded cuDNN version 91002


>> Built 25/408 | usable: 25 | elapsed: 0.673 min
>> Built 50/408 | usable: 50 | elapsed: 1.218 min
>> Built 75/408 | usable: 75 | elapsed: 1.779 min
>> Built 100/408 | usable: 100 | elapsed: 2.36 min
>> Built 125/408 | usable: 125 | elapsed: 2.919 min


I0000 00:00:1777048347.740289     359 service.cc:152] XLA service 0x7a7f24006860 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777048347.740331     359 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1777048347.740335     359 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1777048350.996370     359 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


>> Built 150/408 | usable: 150 | elapsed: 3.555 min
>> Built 175/408 | usable: 175 | elapsed: 4.108 min
>> Built 200/408 | usable: 200 | elapsed: 4.674 min
>> Built 225/408 | usable: 225 | elapsed: 5.234 min
>> Built 250/408 | usable: 250 | elapsed: 5.803 min
>> Built 275/408 | usable: 275 | elapsed: 6.366 min
>> Built 300/408 | usable: 300 | elapsed: 6.931 min
>> Built 325/408 | usable: 325 | elapsed: 7.488 min
>> Built 350/408 | usable: 350 | elapsed: 8.051 min
>> Built 375/408 | usable: 375 | elapsed: 8.609 min
>> Built 400/408 | usable: 400 | elapsed: 9.16 min
Total persons to check: 389
Similarity threshold: 0.65
[1/389] n000033
Source folder    : val
Status           : NEW - added
Best match       : n000154
Best similarity  : 0.3887
Threshold        : 0.65
Total persons    : 409
Elapsed          : 0.023 minutes
[2/389] n000488
Source folder    : val
Status           : NEW - added
Best match       : n000011
Best similarity  : 0.4457
Threshold        : 0.65
Total persons    : 410
E

In [27]:
# check n.duplicates found
total = len(os.listdir(VGG_CLEANED_TRAIN)) + len(os.listdir(VGG_CLEANED_VAL)) + len(os.listdir(LFW_FILTERED)) + len(os.listdir(FAMOUS_PERSONS_DATASET))
unique = len(cleaned_persons)

total - unique

11

## 4.0 Saving

In [34]:
from pathlib import Path

In [32]:
SAVE_FOLDER = "/kaggle/working/final_data_folder"
os.makedirs(SAVE_FOLDER, exist_ok = True)

In [38]:
def save_cleaned_persons(cleaned_persons, save_folder):
    os.makedirs(save_folder, exist_ok = True)

    saved = 0
    skipped = 0

    for person_path in cleaned_persons:
        person_path = Path(person_path)

        if not person_path.exists() or not person_path.is_dir():
            print(f"Skipped invalid path: {person_path}")
            skipped += 1
            continue

        person_name = person_path.name
        dest_path = Path(save_folder) / person_name

        # avoid name conflict
        if dest_path.exists():
            parent_name = person_path.parent.name
            dest_path = Path(save_folder) / f"{parent_name}_{person_name}"

        shutil.copytree(person_path, dest_path)
        saved += 1

        if saved % 50 == 0:
            print(f"saved {saved} persons...")

    print("=" * 80)
    print(f"Done.")
    print(f"saved: {saved}")
    print(f"Skipped: {skipped}")
    print(f"Final folder: {save_folder}")

In [39]:
save_cleaned_persons(cleaned_persons = cleaned_persons, save_folder = SAVE_FOLDER)

saved 50 persons...
saved 100 persons...
saved 150 persons...
saved 200 persons...
saved 250 persons...
saved 300 persons...
saved 350 persons...
saved 400 persons...
saved 450 persons...
saved 500 persons...
saved 550 persons...
saved 600 persons...
saved 650 persons...
saved 700 persons...
saved 750 persons...
Done.
saved: 786
Skipped: 0
Final folder: /kaggle/working/final_data_folder


In [41]:
len(os.listdir(SAVE_FOLDER))

786

In [43]:
shutil.make_archive(
    base_name = "/kaggle/working/final_face_recognition_dataset",
    format = "zip",
    root_dir = SAVE_FOLDER
)

'/kaggle/working/final_face_recognition_dataset.zip'

In [52]:
# saving
UPLOAD_DIR = "/kaggle/working/upload_dataset"
os.makedirs(UPLOAD_DIR, exist_ok = True)

shutil.copy(
    "/kaggle/working/final_face_recognition_dataset.zip",
    os.path.join(UPLOAD_DIR, "data.zip")
)

'/kaggle/working/upload_dataset/data.zip'

In [53]:
metadata = {
    "title": "face-recognition-final",
    "id": "ahmeddragabb/face-recognition-final",  
    "licenses": [{"name": "CC0-1.0"}]
}

with open(os.path.join(UPLOAD_DIR, "dataset-metadata.json"), "w") as f:
    json.dump(metadata, f, indent = 2)

In [54]:
!kaggle datasets create -p /kaggle/working/upload_dataset

Starting upload for file data.zip
100%|███████████████████████████████████████| 1.37G/1.37G [00:14<00:00, 103MB/s]
Upload successful: data.zip (1GB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/ahmeddragabb/face-recognition-final


---
---

***ALHAMDULILLAH***